# UNet → Mask2Former

Постепенное построение Mask2Former архитектуры через наследование, начиная с базовой UNet. Каждый шаг добавляет один ключевой компонент, приближая нас к полноценной реализации.

**Основная идея Mask2Former:** вместо плотных per-pixel предсказаний используем набор learnable queries, каждый из которых ищет один объект через cross-attention к feature map. Это универсальный подход для semantic, instance и panoptic segmentation.

**Ключевые отличия от UNet:**
- UNet: $f: \mathbb{R}^{H \times W \times 3} \to \mathbb{R}^{H \times W \times C}$ — плотные предсказания
- Mask2Former: $f: \mathbb{R}^{H \times W \times 3} \to \{(c_i, m_i)\}_{i=1}^N$ — набор из $N$ пар (класс, маска)

Это позволяет естественно разделять инстансы одного класса.


In [ ]:
import sys
sys.path.insert(0, '../src')
import torch
import torch.nn as nn
import torch.nn.functional as F
from models.unet import UNetResNet50
import math

### Зачем начинать с UNet?

UNet уже умеет извлекать полезные признаки из изображений и генерировать плотные предсказания. У нас есть энкодер (downsampling), который постепенно увеличивает receptive field, и декодер (upsampling), который восстанавливает пространственное разрешение.

Базовый UNet выдаёт per-pixel предсказания: $f: \mathbb{R}^{H \times W \times 3} \to \mathbb{R}^{H \times W \times C}$, где $C$ — число классов. Это хорошо для семантической сегментации, но не умеет разделять инстансы.

Mask2Former работает иначе — он предсказывает набор масок через queries, каждый query отвечает за один объект. Но нам всё ещё нужен мощный backbone для извлечения признаков, и UNet отлично подходит для этой роли.


## Step 0: Base UNet

In [ ]:
class Step0(UNetResNet50):
    pass

model = Step0(num_classes=3, pretrained=False)
out = model(torch.randn(2, 3, 360, 480))
print(f"Step 0: {out.shape}")

### Зачем нужна промежуточная feature map?

Mask2Former использует признаки из middle layers декодера, а не только финальные предсказания. Почему? Потому что нам нужна карта признаков с хорошим балансом:
- Достаточно высокое разрешение (чтобы сохранить детали границ объектов)
- Достаточно семантически богатые признаки (чтобы различать объекты)

Берём выход `dec2` (после двух блоков декодера) и проецируем его в пространство размерности $d=256$:
$$
\text{features} = \text{Conv}_{1 \times 1}(\text{dec2}) \in \mathbb{R}^{B \times d \times H' \times W'}
$$

Эти признаки будем использовать как "память" для transformer и для генерации масок.

Вы спросите: почему иенно второй и почему один? - ответ прост: нет жалания усложнять сейчас архитектуру. На самом деле в нормальном mask2former фичи берутся либо с разных уровней либо с последнего. Но, чтобы это работало и быстро обучалось на вашим графических процессорах предлагаю опустить реализацию deformable и masked attention (которые позволяют эффективно работать с большой фича мапой для классификации) и взять такой уровень фичей, который не сделает cross attention слишком накладным (напомню, что сложность квадратичная от размер фича мапы).


## Step 1: Extract Feature Map

In [ ]:
class Step1(UNetResNet50):
    def __init__(self, num_classes=3, emb_dim=256):
        super().__init__(num_classes, pretrained=False)
        self.emb_dim = emb_dim
        self.feat_proj = nn.Conv2d(512, emb_dim, 1)
    
    def forward(self, x):
        input_size = x.shape[2:]
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(enc1)
        enc3 = self.encoder3(enc2)
        enc4 = self.encoder4(enc3)
        enc5 = self.encoder5(enc4)
        bridge = self.bridge(enc5)
        
        dec1 = self.up1(bridge)
        dec1 = self._match_size(dec1, enc4)
        dec1 = torch.cat([dec1, enc4], dim=1)
        dec1 = self.dec1(dec1)
        
        dec2 = self.up2(dec1)
        dec2 = self._match_size(dec2, enc3)
        dec2 = torch.cat([dec2, enc3], dim=1)
        dec2 = self.dec2(dec2)
        
        features = self.feat_proj(dec2)
        
        dec3 = self.up3(dec2)
        dec3 = self._match_size(dec3, enc2)
        dec3 = torch.cat([dec3, enc2], dim=1)
        dec3 = self.dec3(dec3)
        
        dec4 = self.up4(dec3)
        dec4 = self._match_size(dec4, enc1)
        dec4 = torch.cat([dec4, enc1], dim=1)
        dec4 = self.dec4(dec4)
        
        dec5 = self.up5(dec4)
        dec5 = self.dec5(dec5)
        output = self.final(dec5)
        output = F.interpolate(output, size=input_size, mode='bilinear', align_corners=False)
        
        return output, features

model = Step1(num_classes=3, emb_dim=256)
out, feat = model(torch.randn(2, 3, 360, 480))
print(f"Step 1: out={out.shape}, features={feat.shape}")

### Переход к query-based подходу

Это ключевая идея DETR и Mask2Former: вместо того чтобы предсказывать плотную карту классов для каждого пикселя, мы создаём фиксированное число learnable queries $Q \in \mathbb{R}^{N \times d}$, где $N=100$ — обычно совпадает с максимальным количествов объектов на изображении + небольшой запасик.

Каждый query — это learned embedding, который будет "искать" один объект на изображении. На выходе каждый query предсказывает:
- Класс объекта: $p_i \in \mathbb{R}^{C+1}$ (дополнительный класс "no object")
- Маску объекта (добавим позже)

Пока просто пропускаем queries через линейный слой:
$$
\text{class\_logits} = \text{Linear}(Q) \in \mathbb{R}^{B \times N \times (C+1)}
$$

Queries пока статичные (не смотрят на изображение), но скоро добавим transformer, чтобы они стали content-aware. Так что не пугайтесь, что вам тут что-то не понятно :), мы еще не начали насыщать запросы фичами с картинки просто.


## Step 2: Add Queries

In [ ]:
class Step2(Step1):
    def __init__(self, num_classes=3, emb_dim=256, num_queries=100):
        super().__init__(num_classes, emb_dim)
        self.num_queries = num_queries
        self.queries = nn.Embedding(num_queries, emb_dim)
        self.class_head = nn.Linear(emb_dim, num_classes + 1)
    
    def forward(self, x):
        B = x.shape[0]
        semantic_out, features = super().forward(x)
        
        queries = self.queries.weight.unsqueeze(0).expand(B, -1, -1)
        class_logits = self.class_head(queries)
        
        return {'semantic': semantic_out, 'class_logits': class_logits, 'features': features}

model = Step2(num_classes=3)
out = model(torch.randn(2, 3, 360, 480))
print(f"Step 2: semantic={out['semantic'].shape}, class_logits={out['class_logits'].shape}")

### Почему transformer нужны позиционные эмбеддинги?

Transformer изначально придуман для последовательностей (слов в тексте), где нет пространственной структуры. Attention механизм сам по себе permutation-invariant — он не знает, где в пространстве находится каждый элемент.

Для изображений это проблема: пиксель в левом верхнем углу должен иметь другое представление, чем пиксель справа внизу, даже если их признаки одинаковые.

Решение — добавить sinusoidal positional encoding (как в Transformer):
$$
\text{PE}(x, y, 2i) = \sin\left(\frac{x}{10000^{2i/d}}\right), \quad \text{PE}(x, y, 2i+1) = \cos\left(\frac{x}{10000^{2i/d}}\right)
$$

Делаем это отдельно для $x$ и $y$, потом конкатенируем. Получаем $\text{pos\_emb} \in \mathbb{R}^{B \times d \times H' \times W'}$.

Также создаём learnable позиционные embeddings для queries (каждый query должен знать свою позицию в наборе, а то как мы их и они друг друга будут отличать, правда? :) ).


## Step 3: Add Positional Encoding

In [ ]:
class PositionEmbedding2D(nn.Module):
    def __init__(self, dim=256, temperature=10000):
        super().__init__()
        self.dim = dim
        self.temperature = temperature
    
    def forward(self, x):
        B, C, H, W = x.shape
        y_embed = torch.arange(H, dtype=torch.float32, device=x.device).view(H, 1).repeat(1, W)
        x_embed = torch.arange(W, dtype=torch.float32, device=x.device).view(1, W).repeat(H, 1)
        
        dim_t = torch.arange(self.dim // 4, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * dim_t / (self.dim // 4))
        
        pos_x = x_embed[:, :, None] / dim_t
        pos_y = y_embed[:, :, None] / dim_t
        
        pos_x = torch.cat([pos_x.sin(), pos_x.cos()], dim=-1)
        pos_y = torch.cat([pos_y.sin(), pos_y.cos()], dim=-1)
        
        pos = torch.cat([pos_y, pos_x], dim=-1).permute(2, 0, 1).unsqueeze(0).expand(B, -1, -1, -1)
        return pos

class Step3(Step2):
    def __init__(self, num_classes=3, emb_dim=256, num_queries=100):
        super().__init__(num_classes, emb_dim, num_queries)
        self.pos_emb = PositionEmbedding2D(emb_dim)
        self.query_pos = nn.Embedding(num_queries, emb_dim)
    
    def forward(self, x):
        out = super().forward(x)
        features = out['features']
        pos_emb = self.pos_emb(features)
        
        out['pos_emb'] = pos_emb
        out['query_pos'] = self.query_pos.weight
        return out

model = Step3(num_classes=3)
out = model(torch.randn(2, 3, 360, 480))
print(f"Step 3: positional encoding added")

### Добавляем Transformer Decoder

Теперь самое интересное: позволим queries смотреть на изображение и обновлять себя через cross-attention. Используем стандартный TransformerDecoder из PyTorch.

Как это работает:
1. Flatten feature map в последовательность: $\text{mem} \in \mathbb{R}^{B \times (H' \cdot W') \times d}$
2. Добавляем positional encoding: $\text{mem} + \text{pos\_emb}$
3. Queries тоже получают свои позиции: $Q + Q_{\text{pos}}$
4. Пропускаем через Transformer Decoder:
   $$
   Q' = \text{TransformerDecoder}(Q + Q_{\text{pos}}, \text{mem} + \text{pos\_emb})
   $$

Внутри каждого decoder layer происходит:
- Self-attention между queries (queries обмениваются информацией)
- Cross-attention queries → memory (queries смотрят на feature map)
- Feed-forward network

После этого queries становятся content-aware — каждый query научился фокусироваться на своём объекте. Красота?)


## Step 4: Add Transformer

In [ ]:
class Step4(Step3):
    def __init__(self, num_classes=3, emb_dim=256, num_queries=100, nhead=8, nlayers=3):
        super().__init__(num_classes, emb_dim, num_queries)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=emb_dim, 
            nhead=nhead, 
            dim_feedforward=emb_dim*4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=nlayers)
    
    def forward(self, x):
        B = x.shape[0]
        out = super().forward(x)
        
        features = out['features']
        pos_emb = out['pos_emb']
        
        mem = features.flatten(2).permute(0,2,1)
        mem_pos = pos_emb.flatten(2).permute(0,2,1)
        
        queries = self.queries.weight.unsqueeze(0).expand(B,-1,-1)
        q_pos = self.query_pos.weight.unsqueeze(0).expand(B,-1,-1)
        
        queries = self.transformer(queries + q_pos, mem + mem_pos)
        
        class_logits = self.class_head(queries)
        out['class_logits'] = class_logits
        return out

model = Step4(num_classes=3, nlayers=3)
out = model(torch.randn(2, 3, 360, 480))
print(f"Step 4: transformer applied, class_logits={out['class_logits'].shape}")

### Генерация масок через einsum

Теперь каждый query должен предсказать свою маску. Идея простая: query содержит семантическую информацию об объекте, а feature map содержит пространственную информацию. Нужно их скомбинировать.

Пропускаем каждый query через linear projection:
$$
\text{mask\_embed}_i = \text{Linear}(Q'_i) \in \mathbb{R}^d
$$

Потом делаем dot product между mask embedding и каждым пикселем feature map:
$$
M_{i,h,w} = \langle \text{mask\_embed}_i, \text{features}_{:,h,w} \rangle
$$

В коде это элегантно записывается через einsum:
$$
\text{masks} = \text{einsum}('bqc,bchw \to bqhw', \text{mask\_embed}, \text{features})
$$

Получаем $N=100$ масок размера $H \times W$. Каждая маска — это логиты (применим sigmoid при подсчёте loss).

Финальный upsample до исходного разрешения через bilinear interpolation.

Вот мы и научились предсказывать маски обьектов и сделали нашу модель универсальной в задаче сегментации. Гц вас

## Step 5: Per-Query Masks

In [ ]:
class Step5(Step4):
    def __init__(self, num_classes=3, emb_dim=256, num_queries=100, nhead=8, nlayers=3):
        super().__init__(num_classes, emb_dim, num_queries, nhead, nlayers)
        self.mask_embed = nn.Linear(emb_dim, emb_dim)
    
    def forward(self, x):
        B = x.shape[0]
        semantic_out, features = Step1.forward(self, x)
        pos_emb = self.pos_emb(features)
        
        mem = features.flatten(2).permute(0,2,1)
        mem_pos = pos_emb.flatten(2).permute(0,2,1)
        
        queries = self.queries.weight.unsqueeze(0).expand(B,-1,-1)
        q_pos = self.query_pos.weight.unsqueeze(0).expand(B,-1,-1)
        
        queries = self.transformer(queries + q_pos, mem + mem_pos)
        
        mask_embed = self.mask_embed(queries)
        masks = torch.einsum('bqc,bchw->bqhw', mask_embed, features)
        masks = F.interpolate(masks, x.shape[-2:], mode='bilinear', align_corners=False)
        
        return {
            'semantic': semantic_out,
            'class_logits': self.class_head(queries),
            'masks': masks
        }

model = Step5(num_classes=3)
out = model(torch.randn(2, 3, 360, 480))
print(f"Step 5: per-query masks={out['masks'].shape}")

### Deep Supervision для ускорения обучения

Deep supervision — это техника, когда мы считаем loss не только на финальном выходе, но и на промежуточных слоях. Зачем?

1. Помогает градиентам лучше проходить через глубокую сеть (каждый слой получает прямой supervision signal)
2. Заставляет ранние слои transformer сразу учиться предсказывать разумные маски
3. Даёт регуляризацию — модель не может полагаться только на последний слой

Модифицируем TransformerDecoder, чтобы он возвращал выходы всех промежуточных слоёв:
$$
[Q^{(1)}, Q^{(2)}, \ldots, Q^{(L)}]
$$

Для каждого $Q^{(l)}$ предсказываем маски и классы, и суммируем все losses:
$$
\mathcal{L} = \sum_{l=1}^{L} \mathcal{L}(Q^{(l)}, \text{targets})
$$

Финальный выход — это всё равно последний слой $Q^{(L)}$, но обучение идёт быстрее и стабильнее.

Это самый легкий прием, который дешево обходится нам во время обучения, но который позволяет быстрее сходится)


## Step 6: Deep Supervision

In [ ]:
class TransformerDecoderWithIntermediates(nn.TransformerDecoder):
    def forward(self, tgt, memory, return_intermediate=False):
        output = tgt
        intermediates = []
        
        for mod in self.layers:
            output = mod(output, memory)
            if return_intermediate:
                intermediates.append(output)
        
        if self.norm is not None:
            output = self.norm(output)
        
        return torch.stack(intermediates) if return_intermediate else output

class Step6(Step5):
    def __init__(self, num_classes=3, emb_dim=256, num_queries=100, nhead=8, nlayers=3):
        super().__init__(num_classes, emb_dim, num_queries, nhead, nlayers)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=emb_dim, 
            nhead=nhead, 
            dim_feedforward=emb_dim*4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = TransformerDecoderWithIntermediates(decoder_layer, num_layers=nlayers)
    
    def forward(self, x):
        B = x.shape[0]
        semantic_out, features = Step1.forward(self, x)
        pos_emb = self.pos_emb(features)
        
        mem = features.flatten(2).permute(0,2,1)
        mem_pos = pos_emb.flatten(2).permute(0,2,1)
        
        queries = self.queries.weight.unsqueeze(0).expand(B,-1,-1)
        q_pos = self.query_pos.weight.unsqueeze(0).expand(B,-1,-1)
        
        intermediate_queries = self.transformer(queries + q_pos, mem + mem_pos, return_intermediate=True)
        
        outputs = []
        for layer_queries in intermediate_queries:
            mask_embed = self.mask_embed(layer_queries)
            masks = torch.einsum('bqc,bchw->bqhw', mask_embed, features)
            masks = F.interpolate(masks, x.shape[-2:], mode='bilinear', align_corners=False)
            outputs.append({'class_logits': self.class_head(layer_queries), 'masks': masks})
        
        final = outputs[-1]
        final['aux_outputs'] = outputs[:-1]
        final['semantic'] = semantic_out
        return final

model = Step6(num_classes=3, nlayers=3)
out = model(torch.randn(2, 3, 360, 480))
print(f"Step 6: {len(out['aux_outputs'])} aux outputs")
print(f"Output: class_logits={out['class_logits'].shape}, masks={out['masks'].shape}")

### Hungarian Matching: как сопоставить предсказания с ground truth?

Ну что народ, модель готова, теперь давайте лосс к ней прикрутим. Как вы увидете сейчас и на занятии по детекции это не тривиальная задача)

У нас есть $N=100$ queries, но в изображении только несколько объектов (скажем, 3). Какой query должен предсказывать какой объект? Это проблема bipartite matching.

Решение: Hungarian algorithm (алгоритм Куна-Манкреса). Идея:
1. Строим cost matrix $C \in \mathbb{R}^{N \times M}$, где $M$ — число GT объектов
2. $C_{ij}$ — стоимость назначения query $i$ на GT объект $j$
3. Ищем оптимальное назначение, минимизирующее суммарную стоимость

Cost состоит из двух компонентов:
$$
C_{ij} = \lambda_{\text{cls}} \cdot \mathcal{L}_{\text{cls}}(p_i, y_j) + \lambda_{\text{dice}} \cdot \mathcal{L}_{\text{dice}}(m_i, g_j)
$$

где:
- $\mathcal{L}_{\text{cls}}$ — classification cost (negative probability правильного класса)
- $\mathcal{L}_{\text{dice}}$ — Dice coefficient между предсказанной маской $m_i$ и GT маской $g_j$

После matching знаем, какие queries matched (их будет $M$ штук), остальные $N-M$ queries должны предсказывать "no object".


## Step 7: Add Hungarian Matcher

In [ ]:
from scipy.optimize import linear_sum_assignment

class Step7(Step6):
    def __init__(self, num_classes=3, emb_dim=256, num_queries=100, nhead=8, nlayers=3):
        super().__init__(num_classes, emb_dim, num_queries, nhead, nlayers)
        self.cost_class = 1.0
        self.cost_dice = 1.0
    
    @torch.no_grad()
    def match(self, outputs, targets):
        B, Q = outputs['class_logits'].shape[:2]
        class_probs = F.softmax(outputs['class_logits'], dim=-1)
        mask_probs = torch.sigmoid(outputs['masks'])
        
        indices = []
        for b in range(B):
            if len(targets[b]['labels']) == 0:
                indices.append((torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)))
                continue
            
            gt_labels = targets[b]['labels']
            gt_masks = targets[b]['masks']
            
            cost_class = -class_probs[b][:, gt_labels]
            
            pred_flat = mask_probs[b].flatten(1)
            gt_flat = gt_masks.flatten(1)
            intersection = torch.matmul(pred_flat, gt_flat.t())
            union = pred_flat.sum(1, keepdim=True) + gt_flat.sum(1, keepdim=True).t() - intersection
            cost_dice = 1 - (2 * intersection / (union + 1e-8))
            
            cost_matrix = self.cost_class * cost_class + self.cost_dice * cost_dice
            pred_idx, gt_idx = linear_sum_assignment(cost_matrix.cpu().numpy())
            indices.append((torch.tensor(pred_idx, dtype=torch.long), torch.tensor(gt_idx, dtype=torch.long)))
        
        return indices

def prepare_targets(semantic_masks, num_classes):
    targets = []
    B, H, W = semantic_masks.shape
    for b in range(B):
        labels_list, masks_list = [], []
        for cls in range(1, num_classes):
            cls_mask = (semantic_masks[b] == cls)
            if cls_mask.any():
                labels_list.append(cls)
                masks_list.append(cls_mask.float())
        if len(labels_list) > 0:
            targets.append({'labels': torch.tensor(labels_list, dtype=torch.long), 'masks': torch.stack(masks_list)})
        else:
            targets.append({'labels': torch.zeros(0, dtype=torch.long), 'masks': torch.zeros((0, H, W))})
    return targets

model = Step7(num_classes=3)
outputs = model(torch.randn(2, 3, 360, 480))
targets = prepare_targets(torch.randint(0, 3, (2, 360, 480)), num_classes=3)
matches = model.match(outputs, targets)
print(f"Step 7: Matched {[len(m[0]) for m in matches]} queries per image")

### Mask2Former Loss: комбинация трёх компонентов

После matching можем посчитать loss. Используем три компонента:

**1. Classification Loss** — cross-entropy для всех $N$ queries:
$$
\mathcal{L}_{\text{cls}} = -\frac{1}{N} \sum_{i=1}^{N} \log p_i(y_i)
$$
где $y_i$ — matched класс (или "no object" для unmatched queries).

**2. Dice Loss** — только для matched queries:
$$
\mathcal{L}_{\text{dice}} = 1 - \frac{2 |m_i \cap g_i|}{|m_i| + |g_i|}
$$
Dice loss хорош для сегментации, потому что напрямую оптимизирует IoU. Он устойчив к class imbalance (объект может быть маленьким).

**3. Sigmoid Focal Loss** — тоже только для matched queries:
$$
\mathcal{L}_{\text{focal}} = -\alpha (1-p_t)^\gamma \log(p_t)
$$
где $p_t = \sigma(m_i)$. Focal loss down-weights easy examples и фокусируется на hard negatives (пиксели на границах объектов).

Итоговый loss:
$$
\mathcal{L} = \lambda_{\text{cls}} \mathcal{L}_{\text{cls}} + \lambda_{\text{dice}} \mathcal{L}_{\text{dice}} + \lambda_{\text{focal}} \mathcal{L}_{\text{focal}}
$$

С deep supervision суммируем losses со всех слоёв.


## Step 8: Add Loss Computation

In [ ]:
def dice_loss(inputs, targets):
    inputs = torch.sigmoid(inputs).flatten(1)
    targets = targets.flatten(1)
    intersection = (inputs * targets).sum(1)
    union = inputs.sum(1) + targets.sum(1)
    return 1 - (2 * intersection / (union + 1e-8)).mean()

def sigmoid_focal_loss(inputs, targets, alpha=0.25, gamma=2.0):
    prob = torch.sigmoid(inputs)
    ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
    p_t = prob * targets + (1 - prob) * (1 - targets)
    loss = ce_loss * ((1 - p_t) ** gamma)
    if alpha >= 0:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss
    return loss.mean()

class Step8(Step7):
    def __init__(self, num_classes=3, emb_dim=256, num_queries=100, nhead=8, nlayers=3, w_class=2.0, w_dice=5.0, w_focal=5.0):
        super().__init__(num_classes, emb_dim, num_queries, nhead, nlayers)
        self.w_class = w_class
        self.w_dice = w_dice
        self.w_focal = w_focal
    
    def compute_loss(self, outputs, targets):
        indices = self.match(outputs, targets)
        loss = self._compute_single_loss(outputs, targets, indices)
        
        if 'aux_outputs' in outputs:
            for aux_out in outputs['aux_outputs']:
                aux_indices = self.match(aux_out, targets)
                loss += self._compute_single_loss(aux_out, targets, aux_indices)
        
        return loss
    
    def _compute_single_loss(self, outputs, targets, indices):
        B, Q, C = outputs['class_logits'].shape
        target_classes = torch.full((B, Q), self.num_classes, dtype=torch.long, device=outputs['class_logits'].device)
        
        for b, (pred_idx, gt_idx) in enumerate(indices):
            if len(pred_idx) > 0:
                target_classes[b, pred_idx] = targets[b]['labels'][gt_idx]
        
        loss_class = F.cross_entropy(outputs['class_logits'].flatten(0, 1), target_classes.flatten())
        
        loss_dice, loss_focal, num_masks = 0, 0, 0
        for b, (pred_idx, gt_idx) in enumerate(indices):
            if len(pred_idx) == 0:
                continue
            pred_masks = outputs['masks'][b, pred_idx]
            gt_masks = targets[b]['masks'][gt_idx]
            loss_dice += dice_loss(pred_masks, gt_masks)
            loss_focal += sigmoid_focal_loss(pred_masks, gt_masks)
            num_masks += 1
        
        if num_masks > 0:
            loss_dice /= num_masks
            loss_focal /= num_masks
        
        return self.w_class * loss_class + self.w_dice * loss_dice + self.w_focal * loss_focal

model = Step8(num_classes=3)
outputs = model(torch.randn(2, 3, 360, 480))
targets = prepare_targets(torch.randint(0, 3, (2, 360, 480)), num_classes=3)
loss = model.compute_loss(outputs, targets)
print(f"Step 8: Loss = {loss.item():.4f}")

### Done

Мы построили огрызок свежей sota архитектуры Mask2Former из старенькой, но надежной Unet. Текущая реализация на самом деле включает в себя почти все моменты mask2former:

1. **Backbone** — UNet для извлечения признаков
2. **Queries** — learnable embeddings для поиска объектов
3. **Positional Encoding** — 2D sinusoidal позиции для feature map
4. **Transformer Decoder** — cross-attention между queries и признаками
5. **Mask Generation** — per-query маски через einsum
6. **Deep Supervision** — промежуточные losses для всех слоёв
7. **Hungarian Matching** — оптимальное сопоставление предсказаний с GT
8. **Combined Loss** — classification + dice + focal loss

С этим я вас и поздравляю, а мы переносим этот код в src/models/mask2former.py и пробуем обучать этого монстра в соседних ноутбках. 

Пысы: очевидно, что архитектура неполноценна, но не в этом цель этого семинара. Основная задача убрать страх, что новейшие архитектуры это очень сложно. Как видете идеи не сложные и понятные, если их разбирать последовательно.